In [1]:
import pandas as pd

# 1. Carrega a base bruta de internações de 2023
caminho_base = "../data/raw/sih_campinas_2023.csv"
df_sih = pd.read_csv(caminho_base, low_memory=False, dtype=str)

# 2. Analisa a coluna de CEP
if 'CEP' in df_sih.columns:
    total_internacoes = len(df_sih)
    
    # Limpa possíveis CEPs que vieram só com espaços em branco e conta os válidos
    df_sih['CEP'] = df_sih['CEP'].str.strip()
    # Substitui strings vazias por NaN para a contagem ficar exata
    import numpy as np
    df_sih['CEP'] = df_sih['CEP'].replace('', np.nan)
    
    ceps_preenchidos = df_sih['CEP'].notna().sum()
    ceps_vazios = df_sih['CEP'].isna().sum()

    print(f"=== ANÁLISE DE PREENCHIMENTO DO CEP (SIH 2023) ===")
    print(f"Total de internações: {total_internacoes}")
    print(f"Com CEP preenchido: {ceps_preenchidos} ({(ceps_preenchidos/total_internacoes)*100:.2f}%)")
    print(f"Sem CEP: {ceps_vazios} ({(ceps_vazios/total_internacoes)*100:.2f}%)")
    
    print("\n--- TOP 10 CEPs com mais internações ---")
    display(df_sih['CEP'].value_counts().head(10))
else:
    print("A coluna CEP não foi encontrada na base.")

=== ANÁLISE DE PREENCHIMENTO DO CEP (SIH 2023) ===
Total de internações: 58470
Com CEP preenchido: 58470 (100.00%)
Sem CEP: 0 (0.00%)

--- TOP 10 CEPs com mais internações ---


CEP
13060646    1023
13010100     364
13036220     236
13083888     189
13051207     172
13100010     165
13010000     150
13059664     144
13051205     139
13059591     116
Name: count, dtype: int64

In [3]:
import pandas as pd

# 1. Carrega a base de internações de 2023 (caso o Kernel tenha reiniciado)
caminho_base = "../data/raw/sih_campinas_2023.csv"
print("Carregando base de internações...")
df_sih = pd.read_csv(caminho_base, low_memory=False, dtype=str)

# Limpeza e tratamento do CEP
df_sih['CEP'] = df_sih['CEP'].str.strip()

# 2. Pega os 10 CEPs mais comuns
top10_ceps = df_sih['CEP'].value_counts().head(10).index.tolist()

# 3. Função de mapeamento rápido por prefixo (100% offline e instantâneo)
def traduzir_cep_rapido(cep):
    cep_str = ''.join(filter(str.isdigit, str(cep)))
    if cep_str.startswith('13060'): return 'Região Noroeste (Campo Grande / Ouro Verde)'
    elif cep_str.startswith('13010'): return 'Região Central'
    elif cep_str.startswith('13036'): return 'Região Norte (Taquaral / Vila Itapura)'
    elif cep_str.startswith('13083'): return 'Região Leste (Barão Geraldo / Cidade Universitária)'
    elif cep_str.startswith('13051') or cep_str.startswith('13059'): return 'Região Sudoeste (Campo Grande)'
    elif cep_str.startswith('13100'): return 'Região Sul (Souzas / Joaquim Egídio)'
    else: return 'Outras Regiões de Campinas'

# 4. Gera a tabela consolidada dos Top 10
resultados = []
for cep in top10_ceps:
    qtd = len(df_sih[df_sih['CEP'] == cep])
    regiao = traduzir_cep_rapido(cep)
    resultados.append({'CEP': cep, 'Total de Internações': qtd, 'Região Estimada': regiao})

df_teste_rapido = pd.DataFrame(resultados)
print("\n=== TOP 10 CEPs E SUAS REGIÕES EM CAMPINAS ===")
display(df_teste_rapido)

Carregando base de internações...

=== TOP 10 CEPs E SUAS REGIÕES EM CAMPINAS ===


,CEP,Total de Internações,Região Estimada
0,13060646,1023,Região Noroeste (Campo Grande / Ouro Verde)
1,13010100,364,Região Central
2,13036220,236,Região Norte (Taquaral / Vila Itapura)
3,13083888,189,Região Leste (Barão Geraldo / Cidade Universit...
4,13051207,172,Região Sudoeste (Campo Grande)
5,13100010,165,Região Sul (Souzas / Joaquim Egídio)
6,13010000,150,Região Central
7,13059664,144,Região Sudoeste (Campo Grande)
8,13051205,139,Região Sudoeste (Campo Grande)
9,13059591,116,Região Sudoeste (Campo Grande)


In [5]:
import requests

def consultar_nome_cid(codigo_cid):
    """Consulta o nome oficial de um código CID-10 em uma API pública"""
    try:
        # Usando a API pública do cids.io para consulta rápida
        url = f"https://cids.io/api/cid10/{codigo_cid}"
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            dados = response.json()
            # Retorna a descrição oficial da doença
            return dados.get('descricao', f'CID-{codigo_cid}')
    except:
        pass
    
    # Fallback com os principais caso a API esteja instável
    dicionario_fallback = {
        'O800': 'Parto normal espontâneo',
        'Z302': 'Seguimento de esterilização',
        'I64': 'Acidente vascular cerebral (AVC), não especificado como hemorrágico ou isquêmico',
        'J960': 'Insuficiência respiratória aguda',
        'I219': 'Infarto agudo do miocárdio não especificado',
        'J189': 'Pneumonia não especificada',
        'N390': 'Infecção do trato urinário de localização não especificada',
        'A419': 'Septicemia não especificada',
        'O828': 'Outros tipos de parto cesáreo',
        'I500': 'Insuficiência cardíaca congestiva'
    }
    return dicionario_fallback.get(codigo_cid, f'CID-{codigo_cid} (Outras)')

# Exemplo de aplicação na nossa tabela do Top 10:
top_causas['Descrição da Causa'] = top_causas['CID-10'].apply(consultar_nome_cid)

# Exibe a tabela atualizada com os nomes corretos buscados dinamicamente
display(top_causas[['CID-10', 'Descrição da Causa', 'Total de Internações']])

,CID-10,Descrição da Causa,Total de Internações
0,O800,Parto normal espontâneo,2625
1,Z302,Seguimento de esterilização,1435
2,I64,"Acidente vascular cerebral (AVC), não especifi...",1269
3,J960,Insuficiência respiratória aguda,1007
4,I219,Infarto agudo do miocárdio não especificado,921
5,J189,Pneumonia não especificada,913
6,N390,Infecção do trato urinário de localização não ...,867
7,A419,Septicemia não especificada,746
8,O828,Outros tipos de parto cesáreo,676
9,I500,Insuficiência cardíaca congestiva,653
